<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

# 🗂️ State vs Runtime Context

**一句话定位**：两者看起来很像——都让工具能访问对话消息之外的额外数据——但用途完全不同。



</div>

## 📌 Runtime Context

> 对话**开始前你就已经知道**的数据。

- 🔒 **只读** —— agent 可以通过工具读它，但不能修改
- ⏱️ **调用时设置** —— 通过 `context=` 从外部注入
- 💾 **不持久化** —— 调用结束就没了
- 🧑‍💼 **例子**：用户画像、账户设置、权限、功能开关

## 🔄 State

> agent 在对话**过程中发现或积累**的数据。

- ✏️ **可变** —— 工具可以通过 `Command(update={...})` 写入
- 🕐 **对话中途更新** —— 随着 agent 了解更多信息而演化
- 💿 **跨轮次持久化** —— 配合 `checkpointer` 使用时，多次调用之间能保留
- 🛒 **例子**：聊到一半暴露出来的偏好、加入购物车的商品、过程中做的决策

---

## ⚖️ 对比一览

| | 🔵 Runtime Context | 🟢 State |
|---|---|---|
| **谁设置** | 调用方（你） | agent（通过工具） |
| **什么时候** | 调用之前 | 对话过程中 |
| **可变吗** | ❌ 否 | ✅ 是 |
| **跨轮次持久化吗** | ❌ 否 | ✅ 是（配合 checkpointer） |
| **用途** | 提前就知道的用户数据 | 会话过程中发现/更新的数据 |

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：两者是**互补**关系，不是二选一。真实应用通常两个都用——runtime context 用来表达「我提前就知道这个用户的什么信息」，state 用来表达「agent 在这次会话里学到了什么」。

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**易踩坑**：和 `context` 不同，`State` **不能**设默认值 ❌ —— `context_schema` 的字段可以写 `favourite_colour: str = "blue"` 这种默认值，但 `state_schema`（继承自 `AgentState`）的新增字段必须显式提供，否则读取时可能 `KeyError`。

</div>

### ⭐️ 和 `context` 类似，`state` 的值也是由 `tools` 在运行时访问的 ✅

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## ✏️ 写入 State

### `Command` 函数用来**更新 state** 🔄

工具想写 state，不能直接 `return` 一个值——必须 `return Command(update={...})`，在 `update` 字典里同时指定要更新的 state 字段和要追加的 `messages`（通常是一条 `ToolMessage`，用 `runtime.tool_call_id` 把它和这次工具调用对上）。

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    "Successfully updated favourite colour",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "3"}},
)

In [7]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='9f02b5e0-fbe5-4f00-8914-5f277d951920'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 141, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpUCq0rXerJuvILgVqme4gka5rPD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd730-5a29-7ce3-a1d9-1eb0171a9474-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_F8VDv7nWhbLNSDRQUuVLy1Zk', 'type': 'tool_call'}], invalid_

In [8]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='ceae359d-ca3d-424e-818e-b345851ccfc5'),
              AIMessage(content='Hello! I’m here and ready to help with whatever you need. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 142, 'total_tokens': 492, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpabEbPgrRPrB5LG87AufmX0qiku', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dd736-6a36-7550-bb78-add255f7d475-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_toke

## 👀 读取 State

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：工具通过 `runtime.state["字段名"]` 读 state，而不是 `runtime.context`——这是 state 和 context 在工具里访问方式上唯一的接口差异。读不到时要 `try/except KeyError`，因为新对话的 `thread_id` 还没写入过这个字段。

</div>

In [9]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "20"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='7bd2efaf-3188-41c6-9dad-f9d29a2294bd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpbKWNEOBnN5O2XirCNJcDlD4Uyc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd737-19b3-7a32-8f94-ecc2f363c64b-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_q6ntD2bPQPdTr6oKsf8gUZRI', 'type': 'tool_call'}], invalid_

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "20"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='7bd2efaf-3188-41c6-9dad-f9d29a2294bd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpbKWNEOBnN5O2XirCNJcDlD4Uyc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd737-19b3-7a32-8f94-ecc2f363c64b-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_q6ntD2bPQPdTr6oKsf8gUZRI', 'type': 'tool_call'}], invalid_